In [ ]:
from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

# Lab | Natural Language Processing
### SMS: SPAM or HAM

### Let's prepare the environment

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer

- Read Data for the Fraudulent Email Kaggle Challenge
- Reduce the training set to speed up development.

In [ ]:
data = pd.read_csv("..//data//kg_train.csv", encoding='latin-1')
data = data.head(1000)
print(data.shape)
data.fillna("", inplace=True)
data

In [ ]:
data.columns

### Let's divide the training and test set into two partitions

In [ ]:
from sklearn.model_selection import train_test_split

X = data.drop(columns="label")
y = data["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, shuffle=True
)

X_train

## Data Preprocessing

In [ ]:
import string
from nltk.corpus import stopwords
print(string.punctuation)
print(stopwords.words("english")[100:110])
from nltk.stem.snowball import SnowballStemmer
snowball = SnowballStemmer('english')

## Now, we have to clean the html code removing words

- First we remove inline JavaScript/CSS
- Then we remove html comments. This has to be done before removing regular tags since comments can contain '>' characters
- Next we can remove the remaining tags

In [ ]:
import re

def clean_html(text: str) -> str:
    """
    Limpia HTML siguiendo este orden estricto:
    1. Quita <script> y <style> (incluyendo su contenido)
    2. Quita comentarios HTML <!-- ... -->
    3. Quita el resto de las etiquetas HTML
    """
    if not text or not isinstance(text, str):
        return text

    # 1. Eliminar bloques <script> y <style>
    text = re.sub(r'(?is)<script\b[^>]*>.*?</script>', '', text)
    text = re.sub(r'(?is)<style\b[^>]*>.*?</style>', '', text)

    # 2. Eliminar comentarios HTML
    text = re.sub(r'(?is)<!--.*?-->', '', text)

    # 3. Eliminar el resto de las etiquetas HTML
    text = re.sub(r'<[^>]+>', '', text)

    text = re.sub(r'\s+', ' ', text)
    text = text.strip()

    return text

# FIX: apply element-wise on the 'text' Series, NOT on the whole DataFrame
X_train_clean = X_train['text'].apply(clean_html)
X_test_clean  = X_test['text'].apply(clean_html)

X_train_clean.head()

- Remove all the special characters
- Remove numbers
- Remove all single characters
- Remove single characters from the start
- Substitute multiple spaces with single space
- Remove prefixed 'b'
- Convert to Lowercase

In [ ]:
def deep_clean_text(text: str) -> str:
    if not text or not isinstance(text, str):
        return ""

    # 1. Convert to lowercase
    text = text.lower()

    # 2. Remove prefixed b (byte literals)
    text = re.sub(r'^b\s*', '', text)

    # 3. Remove numbers
    text = re.sub(r'\d+', '', text)

    # 4. Remove special characters, keep letters and spaces
    text = re.sub(r'[^a-z\s]', '', text)

    # 5. Remove isolated single characters
    text = re.sub(r'\b[a-z]\b', '', text)

    # 6. Remove single characters from the beginning
    text = re.sub(r'^\s*[a-z]\s+', '', text)

    # 7. Collapse multiple spaces
    text = re.sub(r'\s+', ' ', text)

    # 8. Strip
    text = text.strip()

    return text

X_train_clean = X_train_clean.apply(deep_clean_text)
X_test_clean  = X_test_clean.apply(deep_clean_text)

X_train_clean.head()

## Now let's work on removing stopwords
Remove the stopwords.

In [ ]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

def remove_stopwords_nltk(text):
    if not isinstance(text, str) or not text:
        return ""
    tokens = word_tokenize(text.lower())
    filtered = [word for word in tokens if word not in stop_words]
    return ' '.join(filtered)

X_train_clean = X_train_clean.apply(remove_stopwords_nltk)
X_test_clean  = X_test_clean.apply(remove_stopwords_nltk)

X_train_clean.head()

## Tame Your Text with Lemmatization
Break sentences into words, then use lemmatization to reduce them to their base form (e.g., "running" becomes "run"). See how this creates cleaner data for analysis!

In [ ]:
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet
from nltk import pos_tag

nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')

lemmatizer = WordNetLemmatizer()

def get_wordnet_pos(treebank_tag):
    """Map Penn Treebank POS tag to WordNet POS tag."""
    if treebank_tag.startswith('J'):
        return wordnet.ADJ
    elif treebank_tag.startswith('V'):
        return wordnet.VERB
    elif treebank_tag.startswith('N'):
        return wordnet.NOUN
    elif treebank_tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN  # default

def lemmatize_text(text):
    """Tokenize, POS-tag, then lemmatize using the correct POS."""
    if not isinstance(text, str) or not text:
        return ""
    tokens    = word_tokenize(text)
    pos_tagged = pos_tag(tokens)
    lemmatized = [
        lemmatizer.lemmatize(word, get_wordnet_pos(pos))
        for word, pos in pos_tagged
    ]
    return ' '.join(lemmatized)

X_train_clean = X_train_clean.apply(lemmatize_text)
X_test_clean  = X_test_clean.apply(lemmatize_text)

print("Original :", X_train.iloc[0]['text'][:80])
print("Processed:", X_train_clean.iloc[0])

## Build data_train / data_val with preprocessed_text

In [ ]:
data_train = X_train.copy()
data_train['preprocessed_text'] = X_train_clean.values
data_train['label'] = y_train.values

data_val = X_test.copy()
data_val['preprocessed_text'] = X_test_clean.values
data_val['label'] = y_test.values

data_train.head()

## Bag Of Words
Let's get the 10 top words in ham and spam messages (**EXPLORATORY DATA ANALYSIS**)

In [ ]:
from collections import Counter

spam_texts = data_train.loc[data_train['label'] == 1, 'preprocessed_text']
ham_texts  = data_train.loc[data_train['label'] == 0, 'preprocessed_text']

def top_n_words(texts, n=10):
    all_words = ' '.join(texts).split()
    return pd.DataFrame(Counter(all_words).most_common(n), columns=['word', 'count'])

top_spam = top_n_words(spam_texts)
top_ham  = top_n_words(ham_texts)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].barh(top_spam['word'][::-1], top_spam['count'][::-1], color='tomato')
axes[0].set_title('Top 10 words – SPAM')
axes[0].set_xlabel('Frequency')

axes[1].barh(top_ham['word'][::-1], top_ham['count'][::-1], color='steelblue')
axes[1].set_title('Top 10 words – HAM')
axes[1].set_xlabel('Frequency')

plt.tight_layout()
plt.show()

print("Top 10 SPAM words:\n", top_spam.to_string(index=False))
print("\nTop 10 HAM words:\n",  top_ham.to_string(index=False))

## Extra features

In [ ]:
money_simbol_list = "|".join(["euro","dollar","pound","euro",r"\$"])
suspicious_words  = "|".join(["free","cheap","sex","money","account","bank",
                               "fund","transfer","transaction","win","deposit","password"])

data_train['money_mark']       = data_train['preprocessed_text'].str.contains(money_simbol_list, regex=True) * 1
data_train['suspicious_words'] = data_train['preprocessed_text'].str.contains(suspicious_words, regex=True) * 1
data_train['text_len']         = data_train['preprocessed_text'].apply(len)

data_val['money_mark']         = data_val['preprocessed_text'].str.contains(money_simbol_list, regex=True) * 1
data_val['suspicious_words']   = data_val['preprocessed_text'].str.contains(suspicious_words, regex=True) * 1
data_val['text_len']           = data_val['preprocessed_text'].apply(len)

data_train.head()

## How would work the Bag of Words with Count Vectorizer concept?

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

# Fit on train, transform both splits
count_vectorizer = CountVectorizer(max_features=5000)

X_train_bow = count_vectorizer.fit_transform(data_train['preprocessed_text'])
X_test_bow  = count_vectorizer.transform(data_val['preprocessed_text'])

print("Bag of Words – Train shape:", X_train_bow.shape)
print("Bag of Words – Test shape: ", X_test_bow.shape)
print("\nFirst 20 vocabulary tokens:", count_vectorizer.get_feature_names_out()[:20])

## TF-IDF

- Load the vectorizer
- Vectorize all dataset
- print the shape of the vectorized dataset

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Load the vectorizer
tfidf_vectorizer = TfidfVectorizer(max_features=5000, sublinear_tf=True)

# Vectorize: fit on train, transform both
X_train_tfidf = tfidf_vectorizer.fit_transform(data_train['preprocessed_text'])
X_test_tfidf  = tfidf_vectorizer.transform(data_val['preprocessed_text'])

# Print shape
print("TF-IDF – Train shape:", X_train_tfidf.shape)
print("TF-IDF – Test shape: ", X_test_tfidf.shape)

## And the Train a Classifier?

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

# ── Bag of Words ──────────────────────────────────────────────────────────
nb_bow = MultinomialNB()
nb_bow.fit(X_train_bow, data_train['label'])
y_pred_bow = nb_bow.predict(X_test_bow)

print("=== Bag of Words ===")
print(f"Accuracy: {accuracy_score(data_val['label'], y_pred_bow):.4f}")
print(classification_report(data_val['label'], y_pred_bow, target_names=['HAM', 'SPAM']))

# ── TF-IDF ────────────────────────────────────────────────────────────────
nb_tfidf = MultinomialNB()
nb_tfidf.fit(X_train_tfidf, data_train['label'])
y_pred_tfidf = nb_tfidf.predict(X_test_tfidf)

print("=== TF-IDF ===")
print(f"Accuracy: {accuracy_score(data_val['label'], y_pred_tfidf):.4f}")
print(classification_report(data_val['label'], y_pred_tfidf, target_names=['HAM', 'SPAM']))

### Extra Task - Implement a SPAM/HAM classifier

https://www.kaggle.com/t/b384e34013d54d238490103bc3c360ce

The classifier can not be changed!!! It must be the MultinomialNB with default parameters!

Your task is to **find the most relevant features**.

For example, you can test the following options and check which of them performs better:
- Using "Bag of Words" only
- Using "TF-IDF" only
- Bag of Words + extra flags (money_mark, suspicious_words, text_len)
- TF-IDF + extra flags


You can work with teams of two persons (recommended).

In [ ]:
import scipy.sparse as sp
from sklearn.preprocessing import MinMaxScaler

# MinMaxScaler ensures extra features are non-negative (required by MultinomialNB)
scaler = MinMaxScaler()
extra_train = scaler.fit_transform(data_train[['money_mark', 'suspicious_words', 'text_len']].values)
extra_test  = scaler.transform(data_val[['money_mark', 'suspicious_words', 'text_len']].values)

# Combine sparse text features with dense extra features
X_train_bow_extra   = sp.hstack([X_train_bow,   extra_train])
X_test_bow_extra    = sp.hstack([X_test_bow,    extra_test])
X_train_tfidf_extra = sp.hstack([X_train_tfidf, extra_train])
X_test_tfidf_extra  = sp.hstack([X_test_tfidf,  extra_test])

# Evaluate all four variants
experiments = [
    ("BoW only",             X_train_bow,          X_test_bow),
    ("TF-IDF only",          X_train_tfidf,         X_test_tfidf),
    ("BoW + extra flags",    X_train_bow_extra,     X_test_bow_extra),
    ("TF-IDF + extra flags", X_train_tfidf_extra,   X_test_tfidf_extra),
]

results = []
for name, X_tr, X_te in experiments:
    clf = MultinomialNB()   # default params – no changes allowed!
    clf.fit(X_tr, data_train['label'])
    y_pred = clf.predict(X_te)
    acc = accuracy_score(data_val['label'], y_pred)
    results.append({'Experiment': name, 'Accuracy': round(acc, 4)})
    print(f"\n{'='*50}")
    print(f"Experiment : {name}")
    print(f"Accuracy   : {acc:.4f}")
    print(classification_report(data_val['label'], y_pred, target_names=['HAM','SPAM']))

print("\n" + "="*50)
print("SUMMARY (best first)")
print(pd.DataFrame(results).sort_values('Accuracy', ascending=False).to_string(index=False))